In [ ]:
print('===== NATIVE BIASGRAD BITEXACT V2 - 7 args =====')
print('===== NATIVE BIASGRAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/bias_grad_cpu.cpu.c'; so=ex+'/bias_grad_cpu.so'
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare BiasGrad
import ctypes, numpy as np
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
native.BiasGrad.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_longlong,ctypes.c_int,ctypes.c_int]
native.BiasGrad.restype=ctypes.c_int
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/bias_grad_cpu.so')
tkv.BiasGrad.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_longlong,ctypes.c_int,ctypes.c_int]
tkv.BiasGrad.restype=ctypes.c_int
rng=np.random.default_rng(7)
for n,cols in [(8,32),(16,13),(4,64),(32,7)]:
    gy=rng.standard_normal(n*cols).astype(np.float32)
    g0=rng.standard_normal(cols+8).astype(np.float32)
    a=g0.copy(); c=g0.copy(); off=3
    native.BiasGrad(gy.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), gy.size, a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), a.size, off, n, cols)
    tkv.BiasGrad(gy.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), gy.size, c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), c.size, off, n, cols)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d cols=%d off=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(n,cols,off,d==0,d,cols,np.abs(a[:cols]-c[:cols]).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')